## Constructing the Transformer Decoder

Welcome to the fifth and final lesson in Deconstructing the Transformer Architecture! Congratulations on reaching this milestone: you've journeyed through multi-head attention mechanisms, mastered feed-forward networks and normalization techniques, understood positional encodings, and successfully built a complete Transformer encoder. Now, we tackle the final piece of the puzzle: the Transformer Decoder Layer.

While the encoder processes input sequences to create rich representations, the decoder has a more complex responsibility. It must generate output sequences one token at a time while carefully managing what information it can access. This requires two distinct attention mechanisms: masked self-attention, which prevents the decoder from "peeking ahead" at future tokens, and cross-attention, which allows the decoder to focus on relevant parts of the encoder's output. By the end of this lesson, you'll understand how decoders balance autoregressive generation with encoder context, completing your understanding of the full Transformer architecture.

---

### Understanding the Decoder's Unique Role

The decoder layer operates under fundamentally different constraints than the encoder, reflecting its role in sequence generation rather than sequence understanding. While an encoder can attend to all positions simultaneously because the entire input sequence is available, a decoder must generate outputs step by step, only accessing tokens it has already produced. This autoregressive nature requires a careful balance: the decoder needs enough context to make informed predictions while being prevented from accessing future information that would make training trivial and inference impossible.

This challenge manifests in the decoder's architecture through three distinct sub-layers instead of the encoder's two. The first sub-layer implements masked self-attention, where each position can only attend to earlier positions in the target sequence. The second sub-layer introduces cross-attention, allowing decoder positions to attend to all encoder outputs, enabling the model to focus on relevant source information when generating each target token. The final sub-layer uses the same position-wise feed-forward network as the encoder, processing the combined self and cross-attention information to produce the final representations.

The interplay between these mechanisms is what makes sequence-to-sequence tasks possible: translation, summarization, and dialogue generation all rely on this careful orchestration of attention patterns. Understanding this architecture helps explain why Transformer-based models excel at tasks requiring both comprehension of input context and coherent generation of output sequences.

---

### Building the Decoder Layer Foundation

Let's begin constructing our `TransformerDecoderLayer` by examining its three core attention mechanisms and how they differ from the encoder:

```python
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Transformer Decoder Layer"""
        super(TransformerDecoderLayer, self).__init__()

        # Masked Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)

        # Encoder-Decoder Multi-Head Attention
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)

        # Position-wise Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)

        # Add & Norm layers
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)
        self.add_norm3 = AddNorm(d_model, dropout)
```

The decoder's initialization reveals its increased complexity compared to the encoder. We now have two separate `MultiHeadAttention` modules: `self_attention` for masked self-attention within the target sequence, and `cross_attention` for attending to encoder outputs. The feed-forward network remains unchanged, but we need three `AddNorm` instances instead of two, each handling residual connections and normalization for its respective sub-layer. This separation ensures that each attention mechanism can learn its own normalization parameters, optimizing for the different statistical properties of self-attention versus cross-attention outputs.

---

### Implementing the Three-Stage Forward Pass

The decoder's forward pass implements a carefully orchestrated three-stage process that transforms target sequences while incorporating encoder context:

```python
def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
    """Forward pass through decoder layer"""
    # 1. Masked Multi-Head Self-Attention + Add & Norm
    self_attn_output, self_attn_weights = self.self_attention(x, x, x, self_attention_mask)
    x = self.add_norm1(x, self_attn_output)

    # 2. Encoder-Decoder Multi-Head Attention + Add & Norm
    cross_attn_output, cross_attn_weights = self.cross_attention(
        x, encoder_output, encoder_output, cross_attention_mask)
    x = self.add_norm2(x, cross_attn_output)

    # 3. Position-wise Feed-Forward + Add & Norm
    ff_output = self.feed_forward(x)
    x = self.add_norm3(x, ff_output)

    return x, (self_attn_weights, cross_attn_weights)
```

The first stage implements masked self-attention, where the `self_attention_mask` enforces the autoregressive property by preventing each position from attending to future positions. This mask is typically a lower triangular matrix where position $i$ can only attend to positions $j \le i$. The mechanism ensures that during training, even though the entire target sequence is available, the model learns to generate each token based only on previous context.

The second stage introduces cross-attention, the decoder's most distinctive feature. Here, queries come from the decoder state `x` (what we want to generate), while keys and values come from `encoder_output` (what we're conditioning on). This allows each decoder position to selectively focus on relevant parts of the source sequence. For example, when translating "The cat sat" to "Le chat s'assit," the decoder generating "chat" can focus strongly on "cat" in the encoder output.

The final stage processes these enriched representations through the feed-forward network, applying non-linear transformations to the combined self-context and source-context information. By returning both attention weight matrices, we enable analysis of the model's decision-making process.

---

### Creating the Decoder Stack

To complete our decoder implementation, we need both the causal masking function and the decoder stack that combines multiple layers:

```python
def create_causal_mask(seq_len):
    """Create causal mask for decoder self-attention"""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0)  # (1, seq_len, seq_len)

class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Stack of Transformer Decoder Layers"""
        super(TransformerDecoder, self).__init__()

        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, activation)
            for _ in range(num_layers)
        ])

        self.num_layers = num_layers

    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder stack"""
        all_self_attention_weights = []
        all_cross_attention_weights = []

        for layer in self.layers:
            x, (self_attn_weights, cross_attn_weights) = layer(
                x, encoder_output, self_attention_mask, cross_attention_mask)
            all_self_attention_weights.append(self_attn_weights)
            all_cross_attention_weights.append(cross_attn_weights)

        return x, (all_self_attention_weights, all_cross_attention_weights)
```

The `create_causal_mask` function generates the crucial lower triangular mask using `torch.tril`, where entries above the diagonal are zero. This effectively blocks attention to future positions, enforcing the autoregressive constraint that makes training match inference conditions. The mask is unsqueezed to add a batch dimension, making it compatible with batched operations.

The `TransformerDecoder` stack follows a similar pattern to the encoder but processes the more complex decoder layers with their dual attention mechanisms. Each layer refines both the target sequence representations and the attention patterns connecting source and target sequences. The collection of attention weights from all layers provides insights into how different layers specialize: early layers often focus on local dependencies, while later layers capture more abstract, task-specific patterns.

---

### Validating the Implementation

Let's validate our complete implementation with comprehensive tests that demonstrate both individual decoder functionality and full encoder-decoder interaction. First, we test the decoder layer in isolation:

```python
def test_decoder_layer():
    """Test single Transformer Decoder Layer"""
    print("Testing Transformer Decoder Layer...")

    # Configuration
    batch_size = 2
    src_seq_len = 8
    tgt_seq_len = 6
    d_model = 64
    num_heads = 8
    d_ff = 256

    # Create sample inputs
    torch.manual_seed(42)
    decoder_input = torch.randn(batch_size, tgt_seq_len, d_model)
    encoder_output = torch.randn(batch_size, src_seq_len, d_model)

    # Create causal mask for decoder self-attention
    causal_mask = create_causal_mask(tgt_seq_len)

    # Create decoder layer
    decoder_layer = TransformerDecoderLayer(d_model, num_heads, d_ff)

    # Forward pass
    output, (self_attn_weights, cross_attn_weights) = decoder_layer(
        decoder_input, encoder_output, causal_mask)

    print(f"Decoder input shape: {decoder_input.shape}")
    print(f"Encoder output shape: {encoder_output.shape}")
    print(f"Decoder output shape: {output.shape}")
    print(f"Self-attention weights shape: {self_attn_weights.shape}")
    print(f"Cross-attention weights shape: {cross_attn_weights.shape}")

    # Verify dimensions
    assert output.shape == decoder_input.shape
    assert self_attn_weights.shape == (batch_size, num_heads, tgt_seq_len, tgt_seq_len)
    assert cross_attn_weights.shape == (batch_size, num_heads, tgt_seq_len, src_seq_len)

    return decoder_layer
```

Running `test_decoder_layer` produces this output:

```text
Testing Transformer Decoder Layer...
Decoder input shape: torch.Size([2, 6, 64])
Encoder output shape: torch.Size([2, 8, 64])
Decoder output shape: torch.Size([2, 6, 64])
Self-attention weights shape: torch.Size([2, 8, 6, 6])
Cross-attention weights shape: torch.Size([2, 8, 6, 8])
```

These results validate that our decoder correctly preserves target dimensions while processing encoder context. The self-attention weights show the expected 6 × 6 pattern for target positions, while cross-attention weights demonstrate proper source-target interaction with shape 6 × 8.

Now, let's test the complete encoder-decoder pipeline:

```python
def test_encoder_decoder_interaction():
    """Test complete encoder-decoder interaction"""
    print("Testing Encoder-Decoder Interaction...")

    # Configuration
    vocab_size = 1000
    num_layers = 2
    d_model = 64
    num_heads = 8
    d_ff = 256
    src_seq_len = 10
    tgt_seq_len = 8
    batch_size = 2

    # Create sample sequences
    torch.manual_seed(42)
    src_tokens = torch.randint(0, vocab_size, (batch_size, src_seq_len))
    tgt_tokens = torch.randint(0, vocab_size, (batch_size, tgt_seq_len))

    # Create components
    token_embedding = TokenEmbedding(vocab_size, d_model)
    positional_encoding = PositionalEncoding(d_model, dropout=0.1)
    encoder = TransformerEncoder(num_layers, d_model, num_heads, d_ff)
    decoder = TransformerDecoder(num_layers, d_model, num_heads, d_ff)

    # Process source sequence through encoder
    src_embeddings = positional_encoding(token_embedding(src_tokens))
    encoder_output, _ = encoder(src_embeddings)

    # Process target sequence through decoder
    tgt_embeddings = positional_encoding(token_embedding(tgt_tokens))
    causal_mask = create_causal_mask(tgt_seq_len)

    decoder_output, _ = decoder(tgt_embeddings, encoder_output, causal_mask)

    print(f"Source tokens shape: {src_tokens.shape}")
    print(f"Target tokens shape: {tgt_tokens.shape}")
    print(f"Encoder output shape: {encoder_output.shape}")
    print(f"Decoder output shape: {decoder_output.shape}")

    return decoder_output
```

Running `test_encoder_decoder_interaction` produces this output:

```text
Testing Encoder-Decoder Interaction...
Source tokens shape: torch.Size([2, 10])
Target tokens shape: torch.Size([2, 8])
Encoder output shape: torch.Size([2, 10, 64])
Decoder output shape: torch.Size([2, 8, 64])
```

This demonstrates successful end-to-end processing through our complete Transformer implementation, validating that raw token sequences are properly transformed through embeddings, encoders, and decoders while respecting all architectural constraints.

---

### Conclusion and Next Steps

Congratulations on completing the final lesson of Deconstructing the Transformer Architecture! You've successfully built a complete Transformer decoder layer, mastering the intricate dance between masked self-attention and cross-attention that enables powerful sequence generation. This achievement represents a deep understanding of one of the most sophisticated architectures in modern AI, from individual attention mechanisms to complete encoder-decoder systems capable of tackling complex sequence-to-sequence tasks.

Your journey continues with the practice exercises ahead, where you'll apply this architectural knowledge hands-on. Looking forward, the next course in our learning path is "Bringing Transformers to Life: Training & Inference", where we'll combine all these Transformer components into complete models, prepare synthetic datasets, implement training loops, and explore real-world deployment strategies. Get ready to see your architectural mastery come alive through practical model development!

## Travel Through Transformers!

Travel through Transformers! This simulation allows you to visually interact with the Transformer architecture. You can click on specific blocks (sublayers) to bring up additional information about each block, such as inputs, parameters, outputs and operations. Enjoy this journey through the Transformer architecture!


## Building Your First Decoder Layer

Now that you've learned about the Transformer Decoder architecture, it's time to implement its most critical component: masked self-attention.

---

### Your Task

In the provided code, you'll find a `TransformerDecoderLayer` class in which the `forward` method is missing its first stage. Your job is to complete the masked self-attention and the first Add & Norm operation.

You need to:

- Call `self.self_attention` with the correct inputs (remember: for self-attention, query, key, and value are all the input `x`)
- Use `self.add_norm1` to apply the residual connection and layer normalization
- Don't forget to pass the `self_attention_mask` parameter

This is the foundation that makes autoregressive generation possible!

---

### Starter Code

```python
import torch
import torch.nn as nn

from transformer.attention import MultiHeadAttention
from transformer.feed_forward import PositionwiseFeedForward
from transformer.utils import AddNorm

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Transformer Decoder Layer"""
        super(TransformerDecoderLayer, self).__init__()
        
        # Masked Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Encoder-Decoder Multi-Head Attention
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Position-wise Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)
        
        # Add & Norm layers
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)
        self.add_norm3 = AddNorm(d_model, dropout)
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder layer"""
        # TODO: Implement masked self-attention and first Add & Norm
        
        # 2. Encoder-Decoder Multi-Head Attention + Add & Norm
        cross_attn_output, cross_attn_weights = self.cross_attention(
            x, encoder_output, encoder_output, cross_attention_mask)
        x = self.add_norm2(x, cross_attn_output)
        
        # 3. Position-wise Feed-Forward + Add & Norm
        ff_output = self.feed_forward(x)
        x = self.add_norm3(x, ff_output)
        
        return x, (self_attn_weights, cross_attn_weights)

class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Stack of Transformer Decoder Layers"""
        super(TransformerDecoder, self).__init__()
        
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, activation)
            for _ in range(num_layers)
        ])
        
        self.num_layers = num_layers
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder stack"""
        all_self_attention_weights = []
        all_cross_attention_weights = []
        
        for layer in self.layers:
            x, (self_attn_weights, cross_attn_weights) = layer(
                x, encoder_output, self_attention_mask, cross_attention_mask)
            all_self_attention_weights.append(self_attn_weights)
            all_cross_attention_weights.append(cross_attn_weights)
        
        return x, (all_self_attention_weights, all_cross_attention_weights)
```

---

### Solution

Only the first stage of `TransformerDecoderLayer.forward` was missing:

```python
# 1. Masked Multi-Head Self-Attention + Add & Norm
self_attn_output, self_attn_weights = self.self_attention(x, x, x, self_attention_mask)
x = self.add_norm1(x, self_attn_output)
```

- `self.self_attention(x, x, x, self_attention_mask)` passes the same tensor as query, key and value, which makes it self-attention over the target sequence. The `self_attention_mask` (a lower triangular causal mask) stops each position from attending to later positions.
- The attention module returns two values. Both are unpacked: `self_attn_output` goes into the residual connection, and `self_attn_weights` is kept because the existing `return x, (self_attn_weights, cross_attn_weights)` uses it.
- `self.add_norm1(x, self_attn_output)` adds the original `x` to the attention output (residual connection) and applies layer normalization. The result replaces `x`, so the cross-attention stage that follows receives the normalized representation.
- Without the fix, `self_attn_weights` is never defined, so the `return` statement fails with an `UnboundLocalError`. Even if it were defined, `x` would skip masked self-attention entirely and the decoder could not enforce the autoregressive property.

The starter code imports `MultiHeadAttention`, `PositionwiseFeedForward` and `AddNorm` from CodeSignal's `transformer` package, which does not exist in this notebook. The cell below defines them inline, using the versions built in the earlier lessons of this course. It also adds a short check at the end, which is not part of the starter. With a causal mask, the check should print a decoder output shape of `[2, 6, 64]`, self-attention weights of shape `[2, 8, 6, 6]` and cross-attention weights of shape `[2, 8, 6, 8]`.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# The `transformer` package is CodeSignal-only, so the building blocks are defined inline.

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """Multi-Head Attention module"""
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier uniform initialization"""
        for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
            nn.init.xavier_uniform_(linear.weight)
            if linear.bias is not None:
                nn.init.constant_(linear.bias, 0)
    
    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """Scaled dot-product attention for multiple heads"""
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, v)
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        """Forward pass for multi-head attention"""
        batch_size, seq_len_q = query.size(0), query.size(1)
        seq_len_k = key.size(1)
        
        # Linear projections and reshape for multi-head attention
        Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        
        # Adjust mask for multi-head attention
        if mask is not None:
            if mask.dim() == 2:  # [seq_len, seq_len]
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
            elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
                mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]
        
        # Apply attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, seq_len_q, self.d_model)
        output = self.w_o(attention_output)
        
        return output, attention_weights


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1, activation='relu'):
        """Position-wise Feed-Forward Network"""
        super(PositionwiseFeedForward, self).__init__()
        
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Choose activation function
        if activation.lower() == 'relu':
            self.activation = F.relu
        elif activation.lower() == 'gelu':
            self.activation = F.gelu
        else:
            raise ValueError(f"Unsupported activation: {activation}")
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights"""
        nn.init.xavier_uniform_(self.linear1.weight)
        nn.init.xavier_uniform_(self.linear2.weight)
        nn.init.constant_(self.linear1.bias, 0)
        nn.init.constant_(self.linear2.bias, 0)
    
    def forward(self, x):
        """Forward pass through position-wise FFN"""
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x


class AddNorm(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        """Add & Norm layer: residual connection + layer normalization"""
        super(AddNorm, self).__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, sublayer_output):
        """Apply residual connection and layer normalization"""
        output = x + self.dropout(sublayer_output)
        return self.layer_norm(output)


class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Transformer Decoder Layer"""
        super(TransformerDecoderLayer, self).__init__()
        
        # Masked Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Encoder-Decoder Multi-Head Attention
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Position-wise Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)
        
        # Add & Norm layers
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)
        self.add_norm3 = AddNorm(d_model, dropout)
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder layer"""
        # 1. Masked Multi-Head Self-Attention + Add & Norm
        self_attn_output, self_attn_weights = self.self_attention(x, x, x, self_attention_mask)  # x is query, key and value
        x = self.add_norm1(x, self_attn_output)  # residual + layer norm
        
        # 2. Encoder-Decoder Multi-Head Attention + Add & Norm
        cross_attn_output, cross_attn_weights = self.cross_attention(
            x, encoder_output, encoder_output, cross_attention_mask)
        x = self.add_norm2(x, cross_attn_output)
        
        # 3. Position-wise Feed-Forward + Add & Norm
        ff_output = self.feed_forward(x)
        x = self.add_norm3(x, ff_output)
        
        return x, (self_attn_weights, cross_attn_weights)

class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Stack of Transformer Decoder Layers"""
        super(TransformerDecoder, self).__init__()
        
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, activation)
            for _ in range(num_layers)
        ])
        
        self.num_layers = num_layers
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder stack"""
        all_self_attention_weights = []
        all_cross_attention_weights = []
        
        for layer in self.layers:
            x, (self_attn_weights, cross_attn_weights) = layer(
                x, encoder_output, self_attention_mask, cross_attention_mask)
            all_self_attention_weights.append(self_attn_weights)
            all_cross_attention_weights.append(cross_attn_weights)
        
        return x, (all_self_attention_weights, all_cross_attention_weights)


# Quick check (not part of the starter code)
def create_causal_mask(seq_len):
    """Create causal mask for decoder self-attention"""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0)  # (1, seq_len, seq_len)

torch.manual_seed(42)
decoder_input = torch.randn(2, 6, 64)
encoder_output = torch.randn(2, 8, 64)
decoder_layer = TransformerDecoderLayer(d_model=64, num_heads=8, d_ff=256)
output, (self_attn_weights, cross_attn_weights) = decoder_layer(
    decoder_input, encoder_output, create_causal_mask(6))
print(f"Decoder output shape: {output.shape}")
print(f"Self-attention weights shape: {self_attn_weights.shape}")
print(f"Cross-attention weights shape: {cross_attn_weights.shape}")


## Complete the Missing Connection

Now that you've implemented the masked self-attention mechanism, it's time to connect the decoder to the encoder through cross-attention. In this exercise, you'll complete the second stage of the decoder's forward pass by wiring up the encoder-decoder cross-attention mechanism.

The cross-attention mechanism allows the decoder to attend to the encoder's output while generating sequences. Unlike self-attention, where queries, keys, and values all come from the same input, cross-attention uses the decoder's current state as queries while obtaining keys and values from the encoder output.

---

### Your Task

Your job is to complete the missing code in the `forward` method. You'll need to:

- Call the cross-attention mechanism with the correct input configuration
- Apply the appropriate Add & Norm operation afterward

Pay close attention to which inputs serve as queries, keys, and values — this is the key insight that makes cross-attention work. Once you get this right, your decoder will be able to use encoder information to generate much better outputs!

---

### Starter Code

```python
import torch
import torch.nn as nn

from transformer.attention import MultiHeadAttention
from transformer.feed_forward import PositionwiseFeedForward
from transformer.utils import AddNorm

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Transformer Decoder Layer"""
        super(TransformerDecoderLayer, self).__init__()
        
        # Masked Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Encoder-Decoder Multi-Head Attention
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Position-wise Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)
        
        # Add & Norm layers
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)
        self.add_norm3 = AddNorm(d_model, dropout)
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder layer"""
        # 1. Masked Multi-Head Self-Attention + Add & Norm
        self_attn_output, self_attn_weights = self.self_attention(x, x, x, self_attention_mask)
        x = self.add_norm1(x, self_attn_output)
        
        # 2. Encoder-Decoder Multi-Head Attention + Add & Norm
        # TODO: Implement the cross-attention mechanism
        # Hint: The query (Q) comes from x, while key (K) and value (V) come from encoder_output
        # Don't forget to apply add_norm2 after getting the cross-attention output
        
        # 3. Position-wise Feed-Forward + Add & Norm
        ff_output = self.feed_forward(x)
        x = self.add_norm3(x, ff_output)
        
        return x, (self_attn_weights, cross_attn_weights)

class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Stack of Transformer Decoder Layers"""
        super(TransformerDecoder, self).__init__()
        
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, activation)
            for _ in range(num_layers)
        ])
        
        self.num_layers = num_layers
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder stack"""
        all_self_attention_weights = []
        all_cross_attention_weights = []
        
        for layer in self.layers:
            x, (self_attn_weights, cross_attn_weights) = layer(
                x, encoder_output, self_attention_mask, cross_attention_mask)
            all_self_attention_weights.append(self_attn_weights)
            all_cross_attention_weights.append(cross_attn_weights)
        
        return x, (all_self_attention_weights, all_cross_attention_weights)
```

---

### Solution

Only the second stage of `TransformerDecoderLayer.forward` was missing:

```python
# 2. Encoder-Decoder Multi-Head Attention + Add & Norm
cross_attn_output, cross_attn_weights = self.cross_attention(
    x, encoder_output, encoder_output, cross_attention_mask)
x = self.add_norm2(x, cross_attn_output)
```

- `self.cross_attention(x, encoder_output, encoder_output, cross_attention_mask)` passes the decoder state `x` as the query and the encoder output as both key and value. Each target position asks "which source positions are relevant to me?", and the answer is a weighted mix of the encoder's representations.
- The attention module returns two values. Both are unpacked: `cross_attn_output` goes into the residual connection, and `cross_attn_weights` is kept because the existing `return x, (self_attn_weights, cross_attn_weights)` uses it. Its shape is `[batch, heads, tgt_len, src_len]`, so the query length comes from `x` and the key length comes from `encoder_output`.
- `self.add_norm2(x, cross_attn_output)` adds the incoming `x` to the cross-attention output (residual connection) and applies layer normalization. The result replaces `x`, so the feed-forward stage that follows receives the normalized representation. It must be `add_norm2`, since `add_norm1` and `add_norm3` belong to the other two stages.
- Without the fix, `cross_attn_weights` is never defined, so the `return` statement fails with an `UnboundLocalError`. Even if it were defined, the decoder would never see the encoder, so it could not condition its output on the source sequence.

The starter code imports `MultiHeadAttention`, `PositionwiseFeedForward` and `AddNorm` from CodeSignal's `transformer` package, which does not exist in this notebook. The cell below defines them inline, using the versions built in the earlier lessons of this course. The starter has no `main()`, so a short check is added at the end, which is not part of the starter. With a causal mask, it should print a decoder output shape of `[2, 6, 64]`, self-attention weights of shape `[2, 8, 6, 6]` and cross-attention weights of shape `[2, 8, 6, 8]`.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# The `transformer` package is CodeSignal-only, so the building blocks are defined inline.

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """Multi-Head Attention module"""
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier uniform initialization"""
        for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
            nn.init.xavier_uniform_(linear.weight)
            if linear.bias is not None:
                nn.init.constant_(linear.bias, 0)
    
    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """Scaled dot-product attention for multiple heads"""
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, v)
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        """Forward pass for multi-head attention"""
        batch_size, seq_len_q = query.size(0), query.size(1)
        seq_len_k = key.size(1)
        
        # Linear projections and reshape for multi-head attention
        Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        
        # Adjust mask for multi-head attention
        if mask is not None:
            if mask.dim() == 2:  # [seq_len, seq_len]
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
            elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
                mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]
        
        # Apply attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, seq_len_q, self.d_model)
        output = self.w_o(attention_output)
        
        return output, attention_weights


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1, activation='relu'):
        """Position-wise Feed-Forward Network"""
        super(PositionwiseFeedForward, self).__init__()
        
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Choose activation function
        if activation.lower() == 'relu':
            self.activation = F.relu
        elif activation.lower() == 'gelu':
            self.activation = F.gelu
        else:
            raise ValueError(f"Unsupported activation: {activation}")
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights"""
        nn.init.xavier_uniform_(self.linear1.weight)
        nn.init.xavier_uniform_(self.linear2.weight)
        nn.init.constant_(self.linear1.bias, 0)
        nn.init.constant_(self.linear2.bias, 0)
    
    def forward(self, x):
        """Forward pass through position-wise FFN"""
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x


class AddNorm(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        """Add & Norm layer: residual connection + layer normalization"""
        super(AddNorm, self).__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, sublayer_output):
        """Apply residual connection and layer normalization"""
        output = x + self.dropout(sublayer_output)
        return self.layer_norm(output)


class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Transformer Decoder Layer"""
        super(TransformerDecoderLayer, self).__init__()
        
        # Masked Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Encoder-Decoder Multi-Head Attention
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Position-wise Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)
        
        # Add & Norm layers
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)
        self.add_norm3 = AddNorm(d_model, dropout)
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder layer"""
        # 1. Masked Multi-Head Self-Attention + Add & Norm
        self_attn_output, self_attn_weights = self.self_attention(x, x, x, self_attention_mask)
        x = self.add_norm1(x, self_attn_output)
        
        # 2. Encoder-Decoder Multi-Head Attention + Add & Norm
        cross_attn_output, cross_attn_weights = self.cross_attention(
            x, encoder_output, encoder_output, cross_attention_mask)  # Q from x, K and V from the encoder
        x = self.add_norm2(x, cross_attn_output)  # residual + layer norm
        
        # 3. Position-wise Feed-Forward + Add & Norm
        ff_output = self.feed_forward(x)
        x = self.add_norm3(x, ff_output)
        
        return x, (self_attn_weights, cross_attn_weights)

class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Stack of Transformer Decoder Layers"""
        super(TransformerDecoder, self).__init__()
        
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, activation)
            for _ in range(num_layers)
        ])
        
        self.num_layers = num_layers
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder stack"""
        all_self_attention_weights = []
        all_cross_attention_weights = []
        
        for layer in self.layers:
            x, (self_attn_weights, cross_attn_weights) = layer(
                x, encoder_output, self_attention_mask, cross_attention_mask)
            all_self_attention_weights.append(self_attn_weights)
            all_cross_attention_weights.append(cross_attn_weights)
        
        return x, (all_self_attention_weights, all_cross_attention_weights)


# Quick check (not part of the starter code)
def create_causal_mask(seq_len):
    """Create causal mask for decoder self-attention"""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0)  # (1, seq_len, seq_len)

torch.manual_seed(42)
decoder_input = torch.randn(2, 6, 64)
encoder_output = torch.randn(2, 8, 64)
decoder_layer = TransformerDecoderLayer(d_model=64, num_heads=8, d_ff=256)
output, (self_attn_weights, cross_attn_weights) = decoder_layer(
    decoder_input, encoder_output, create_causal_mask(6))
print(f"Decoder output shape: {output.shape}")
print(f"Self-attention weights shape: {self_attn_weights.shape}")
print(f"Cross-attention weights shape: {cross_attn_weights.shape}")


## Assembling Full Decoder Layer

Now that you've mastered the core attention mechanisms, it's time to put together the complete decoder architecture. The previous exercises had you build the masked self-attention and cross-attention components — now it's time to assemble them into a fully functional `TransformerDecoderLayer`.

---

### Your Task

Your job is to complete two key parts of the provided code:

- Initialize the three `AddNorm` layers (`self.add_norm1`, `self.add_norm2`, `self.add_norm3`) in the `__init__` method.
- Implement the complete `forward` method that sequences all three sub-layers: masked self-attention, cross-attention, and feed-forward network, each followed by its corresponding Add & Norm operation.

Remember that the forward pass should return both the final output tensor and the attention weights from both attention mechanisms. This is your chance to see how all the decoder components work together seamlessly!

---

### Starter Code

```python
import torch
import torch.nn as nn

from transformer.attention import MultiHeadAttention
from transformer.feed_forward import PositionwiseFeedForward
from transformer.utils import AddNorm

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Transformer Decoder Layer"""
        super(TransformerDecoderLayer, self).__init__()
        
        # Masked Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Encoder-Decoder Multi-Head Attention
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Position-wise Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)
        
        # TODO: Initialize the three Add & Norm layers (self.add_norm1, self.add_norm2, self.add_norm3)
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder layer"""
        # TODO: Implement the full three-stage forward pass
        pass

class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Stack of Transformer Decoder Layers"""
        super(TransformerDecoder, self).__init__()
        
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, activation)
            for _ in range(num_layers)
        ])
        
        self.num_layers = num_layers
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder stack"""
        all_self_attention_weights = []
        all_cross_attention_weights = []
        
        for layer in self.layers:
            x, (self_attn_weights, cross_attn_weights) = layer(
                x, encoder_output, self_attention_mask, cross_attention_mask)
            all_self_attention_weights.append(self_attn_weights)
            all_cross_attention_weights.append(cross_attn_weights)
        
        return x, (all_self_attention_weights, all_cross_attention_weights)
```

---

### Solution

Two TODOs were missing in `TransformerDecoderLayer`. First, the three `AddNorm` layers in `__init__`:

```python
# Add & Norm layers
self.add_norm1 = AddNorm(d_model, dropout)
self.add_norm2 = AddNorm(d_model, dropout)
self.add_norm3 = AddNorm(d_model, dropout)
```

Second, the full three-stage `forward`, which replaces `pass`:

```python
# 1. Masked Multi-Head Self-Attention + Add & Norm
self_attn_output, self_attn_weights = self.self_attention(x, x, x, self_attention_mask)
x = self.add_norm1(x, self_attn_output)

# 2. Encoder-Decoder Multi-Head Attention + Add & Norm
cross_attn_output, cross_attn_weights = self.cross_attention(
    x, encoder_output, encoder_output, cross_attention_mask)
x = self.add_norm2(x, cross_attn_output)

# 3. Position-wise Feed-Forward + Add & Norm
ff_output = self.feed_forward(x)
x = self.add_norm3(x, ff_output)

return x, (self_attn_weights, cross_attn_weights)
```

- Each sub-layer gets its own `AddNorm` instance, so each one learns its own `LayerNorm` parameters. They all take `d_model` and `dropout`, the same arguments the other modules receive.
- Stage 1 is masked self-attention: `x` is query, key and value, and `self_attention_mask` blocks future positions. Stage 2 is cross-attention: the query is `x`, while the key and value are `encoder_output`. Stage 3 is the feed-forward network.
- Every stage follows the same pattern: run the sub-layer, then `add_normN(x, sublayer_output)` for the residual connection and layer normalization. The result replaces `x`, so each stage feeds the next.
- The return value is a tuple `(x, (self_attn_weights, cross_attn_weights))`. `TransformerDecoder.forward` unpacks exactly this structure with `x, (self_attn_weights, cross_attn_weights) = layer(...)`.
- Without the fix, `__init__` never creates the `add_norm` attributes, and `forward` returns `None` because of `pass`. The decoder stack then fails when it tries to unpack that `None`, with a `TypeError`.

The starter code imports `MultiHeadAttention`, `PositionwiseFeedForward` and `AddNorm` from CodeSignal's `transformer` package, which does not exist in this notebook. The cell below defines them inline, using the versions built in the earlier lessons of this course. The starter has no `main()`, so a short check is added at the end, which is not part of the starter. With a causal mask, it should print a decoder output shape of `[2, 6, 64]`, self-attention weights of shape `[2, 8, 6, 6]` and cross-attention weights of shape `[2, 8, 6, 8]`.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# The `transformer` package is CodeSignal-only, so the building blocks are defined inline.

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """Multi-Head Attention module"""
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier uniform initialization"""
        for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
            nn.init.xavier_uniform_(linear.weight)
            if linear.bias is not None:
                nn.init.constant_(linear.bias, 0)
    
    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """Scaled dot-product attention for multiple heads"""
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, v)
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        """Forward pass for multi-head attention"""
        batch_size, seq_len_q = query.size(0), query.size(1)
        seq_len_k = key.size(1)
        
        # Linear projections and reshape for multi-head attention
        Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        
        # Adjust mask for multi-head attention
        if mask is not None:
            if mask.dim() == 2:  # [seq_len, seq_len]
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
            elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
                mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]
        
        # Apply attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, seq_len_q, self.d_model)
        output = self.w_o(attention_output)
        
        return output, attention_weights


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1, activation='relu'):
        """Position-wise Feed-Forward Network"""
        super(PositionwiseFeedForward, self).__init__()
        
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Choose activation function
        if activation.lower() == 'relu':
            self.activation = F.relu
        elif activation.lower() == 'gelu':
            self.activation = F.gelu
        else:
            raise ValueError(f"Unsupported activation: {activation}")
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights"""
        nn.init.xavier_uniform_(self.linear1.weight)
        nn.init.xavier_uniform_(self.linear2.weight)
        nn.init.constant_(self.linear1.bias, 0)
        nn.init.constant_(self.linear2.bias, 0)
    
    def forward(self, x):
        """Forward pass through position-wise FFN"""
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x


class AddNorm(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        """Add & Norm layer: residual connection + layer normalization"""
        super(AddNorm, self).__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, sublayer_output):
        """Apply residual connection and layer normalization"""
        output = x + self.dropout(sublayer_output)
        return self.layer_norm(output)


class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Transformer Decoder Layer"""
        super(TransformerDecoderLayer, self).__init__()
        
        # Masked Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Encoder-Decoder Multi-Head Attention
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Position-wise Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)
        
        # Add & Norm layers: one per sub-layer
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)
        self.add_norm3 = AddNorm(d_model, dropout)
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder layer"""
        # 1. Masked Multi-Head Self-Attention + Add & Norm
        self_attn_output, self_attn_weights = self.self_attention(x, x, x, self_attention_mask)  # x is query, key and value
        x = self.add_norm1(x, self_attn_output)  # residual + layer norm
        
        # 2. Encoder-Decoder Multi-Head Attention + Add & Norm
        cross_attn_output, cross_attn_weights = self.cross_attention(
            x, encoder_output, encoder_output, cross_attention_mask)  # Q from x, K and V from the encoder
        x = self.add_norm2(x, cross_attn_output)  # residual + layer norm
        
        # 3. Position-wise Feed-Forward + Add & Norm
        ff_output = self.feed_forward(x)
        x = self.add_norm3(x, ff_output)  # residual + layer norm
        
        return x, (self_attn_weights, cross_attn_weights)

class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Stack of Transformer Decoder Layers"""
        super(TransformerDecoder, self).__init__()
        
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, activation)
            for _ in range(num_layers)
        ])
        
        self.num_layers = num_layers
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder stack"""
        all_self_attention_weights = []
        all_cross_attention_weights = []
        
        for layer in self.layers:
            x, (self_attn_weights, cross_attn_weights) = layer(
                x, encoder_output, self_attention_mask, cross_attention_mask)
            all_self_attention_weights.append(self_attn_weights)
            all_cross_attention_weights.append(cross_attn_weights)
        
        return x, (all_self_attention_weights, all_cross_attention_weights)


# Quick check (not part of the starter code)
def create_causal_mask(seq_len):
    """Create causal mask for decoder self-attention"""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0)  # (1, seq_len, seq_len)

torch.manual_seed(42)
decoder_input = torch.randn(2, 6, 64)
encoder_output = torch.randn(2, 8, 64)
decoder_layer = TransformerDecoderLayer(d_model=64, num_heads=8, d_ff=256)
output, (self_attn_weights, cross_attn_weights) = decoder_layer(
    decoder_input, encoder_output, create_causal_mask(6))
print(f"Decoder output shape: {output.shape}")
print(f"Self-attention weights shape: {self_attn_weights.shape}")
print(f"Cross-attention weights shape: {cross_attn_weights.shape}")


## Building the Decoder Stack

Now that you have a complete decoder layer working, it's time to stack multiple layers together to build the full decoder. Your job is to implement the `forward` method for the `TransformerDecoder` class.

The provided code has a complete initialization, but the `forward` method contains only a `pass` statement. You need to process the input through all decoder layers in sequence, collecting attention weights from each layer along the way.

---

### Your Task

- Initialize empty lists to store attention weights
- Loop through each layer in `self.layers`
- Pass the input and encoder output through each layer
- Update the input with each layer's output
- Collect the attention weights from each layer

Complete this implementation, and you'll have a working transformer decoder stack!

---

### Starter Code

```python
import torch
import torch.nn as nn

from transformer.attention import MultiHeadAttention
from transformer.feed_forward import PositionwiseFeedForward
from transformer.utils import AddNorm

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Transformer Decoder Layer"""
        super(TransformerDecoderLayer, self).__init__()
        
        # Masked Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Encoder-Decoder Multi-Head Attention
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Position-wise Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)
        
        # Add & Norm layers
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)
        self.add_norm3 = AddNorm(d_model, dropout)
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder layer"""
        # 1. Masked Multi-Head Self-Attention + Add & Norm
        self_attn_output, self_attn_weights = self.self_attention(x, x, x, self_attention_mask)
        x = self.add_norm1(x, self_attn_output)
        
        # 2. Encoder-Decoder Multi-Head Attention + Add & Norm
        cross_attn_output, cross_attn_weights = self.cross_attention(
            x, encoder_output, encoder_output, cross_attention_mask)
        x = self.add_norm2(x, cross_attn_output)
        
        # 3. Position-wise Feed-Forward + Add & Norm
        ff_output = self.feed_forward(x)
        x = self.add_norm3(x, ff_output)
        
        return x, (self_attn_weights, cross_attn_weights)

class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Stack of Transformer Decoder Layers"""
        super(TransformerDecoder, self).__init__()
        
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, activation)
            for _ in range(num_layers)
        ])
        
        self.num_layers = num_layers
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder stack"""
        # TODO: Implement the forward pass through all decoder layers
        # Initialize empty lists for storing attention weights from each layer
        # Loop through each layer in self.layers and pass the inputs through it
        # Update x with the output and collect the attention weights
        # Return the final x and all collected attention weights
        pass
```

---

### Solution

Only `TransformerDecoder.forward` was missing:

```python
all_self_attention_weights = []
all_cross_attention_weights = []

for layer in self.layers:
    x, (self_attn_weights, cross_attn_weights) = layer(
        x, encoder_output, self_attention_mask, cross_attention_mask)
    all_self_attention_weights.append(self_attn_weights)
    all_cross_attention_weights.append(cross_attn_weights)

return x, (all_self_attention_weights, all_cross_attention_weights)
```

- The two empty lists collect one attention tensor per layer, so after the loop each list has `num_layers` entries.
- `for layer in self.layers` visits the layers in order. Each layer is called with the current `x`, the same `encoder_output`, and both masks, and returns `(x, (self_attn_weights, cross_attn_weights))`, which matches the tuple that `TransformerDecoderLayer.forward` returns.
- Assigning the result back to `x` is what stacks the layers: the output of layer $n$ becomes the input of layer $n+1$. `encoder_output` is not updated, so every layer cross-attends to the same encoder representation.
- The return value is the final `x` plus the two lists of weights, in the `(x, (self_weights_list, cross_weights_list))` structure the lesson uses.
- Without the fix, `forward` returns `None` because of `pass`. A caller such as `decoder_output, _ = decoder(...)` then fails with a `TypeError` (cannot unpack a non-iterable `NoneType`).

The starter code imports `MultiHeadAttention`, `PositionwiseFeedForward` and `AddNorm` from CodeSignal's `transformer` package, which does not exist in this notebook. The cell below defines them inline, using the versions built in the earlier lessons of this course. The starter has no `main()`, so a short check is added at the end, which is not part of the starter. It runs a 2-layer decoder with a causal mask and should print an output shape of `[2, 6, 64]`, 2 self-attention tensors of shape `[2, 8, 6, 6]` and 2 cross-attention tensors of shape `[2, 8, 6, 8]`.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# The `transformer` package is CodeSignal-only, so the building blocks are defined inline.

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        """Multi-Head Attention module"""
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V and output
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights using Xavier uniform initialization"""
        for linear in [self.w_q, self.w_k, self.w_v, self.w_o]:
            nn.init.xavier_uniform_(linear.weight)
            if linear.bias is not None:
                nn.init.constant_(linear.bias, 0)
    
    def scaled_dot_product_attention(self, q, k, v, mask=None):
        """Scaled dot-product attention for multiple heads"""
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, v)
        return output, attention_weights
    
    def forward(self, query, key, value, mask=None):
        """Forward pass for multi-head attention"""
        batch_size, seq_len_q = query.size(0), query.size(1)
        seq_len_k = key.size(1)
        
        # Linear projections and reshape for multi-head attention
        Q = self.w_q(query).view(batch_size, seq_len_q, self.num_heads, self.d_k).transpose(1, 2)
        K = self.w_k(key).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        V = self.w_v(value).view(batch_size, seq_len_k, self.num_heads, self.d_k).transpose(1, 2)
        
        # Adjust mask for multi-head attention
        if mask is not None:
            if mask.dim() == 2:  # [seq_len, seq_len]
                mask = mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]
            elif mask.dim() == 3:  # [batch_size, seq_len, seq_len]
                mask = mask.unsqueeze(1)  # [batch_size, 1, seq_len, seq_len]
        
        # Apply attention
        attention_output, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads and apply output projection
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, seq_len_q, self.d_model)
        output = self.w_o(attention_output)
        
        return output, attention_weights


class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1, activation='relu'):
        """Position-wise Feed-Forward Network"""
        super(PositionwiseFeedForward, self).__init__()
        
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Choose activation function
        if activation.lower() == 'relu':
            self.activation = F.relu
        elif activation.lower() == 'gelu':
            self.activation = F.gelu
        else:
            raise ValueError(f"Unsupported activation: {activation}")
        
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights"""
        nn.init.xavier_uniform_(self.linear1.weight)
        nn.init.xavier_uniform_(self.linear2.weight)
        nn.init.constant_(self.linear1.bias, 0)
        nn.init.constant_(self.linear2.bias, 0)
    
    def forward(self, x):
        """Forward pass through position-wise FFN"""
        x = self.linear1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x


class AddNorm(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        """Add & Norm layer: residual connection + layer normalization"""
        super(AddNorm, self).__init__()
        self.layer_norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, sublayer_output):
        """Apply residual connection and layer normalization"""
        output = x + self.dropout(sublayer_output)
        return self.layer_norm(output)


class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Transformer Decoder Layer"""
        super(TransformerDecoderLayer, self).__init__()
        
        # Masked Multi-Head Self-Attention
        self.self_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Encoder-Decoder Multi-Head Attention
        self.cross_attention = MultiHeadAttention(d_model, num_heads, dropout)
        
        # Position-wise Feed-Forward Network
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout, activation)
        
        # Add & Norm layers: one per sub-layer
        self.add_norm1 = AddNorm(d_model, dropout)
        self.add_norm2 = AddNorm(d_model, dropout)
        self.add_norm3 = AddNorm(d_model, dropout)
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder layer"""
        # 1. Masked Multi-Head Self-Attention + Add & Norm
        self_attn_output, self_attn_weights = self.self_attention(x, x, x, self_attention_mask)  # x is query, key and value
        x = self.add_norm1(x, self_attn_output)  # residual + layer norm
        
        # 2. Encoder-Decoder Multi-Head Attention + Add & Norm
        cross_attn_output, cross_attn_weights = self.cross_attention(
            x, encoder_output, encoder_output, cross_attention_mask)  # Q from x, K and V from the encoder
        x = self.add_norm2(x, cross_attn_output)  # residual + layer norm
        
        # 3. Position-wise Feed-Forward + Add & Norm
        ff_output = self.feed_forward(x)
        x = self.add_norm3(x, ff_output)  # residual + layer norm
        
        return x, (self_attn_weights, cross_attn_weights)

class TransformerDecoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1, activation='relu'):
        """Stack of Transformer Decoder Layers"""
        super(TransformerDecoder, self).__init__()
        
        self.layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, num_heads, d_ff, dropout, activation)
            for _ in range(num_layers)
        ])
        
        self.num_layers = num_layers
    
    def forward(self, x, encoder_output, self_attention_mask=None, cross_attention_mask=None):
        """Forward pass through decoder stack"""
        # Initialize empty lists for storing attention weights from each layer
        all_self_attention_weights = []
        all_cross_attention_weights = []
        
        # Pass the inputs through each decoder layer in sequence
        for layer in self.layers:
            x, (self_attn_weights, cross_attn_weights) = layer(
                x, encoder_output, self_attention_mask, cross_attention_mask)  # x is updated by each layer
            all_self_attention_weights.append(self_attn_weights)  # collect per-layer weights
            all_cross_attention_weights.append(cross_attn_weights)
        
        # Return the final output and all collected attention weights
        return x, (all_self_attention_weights, all_cross_attention_weights)


# Quick check (not part of the starter code)
def create_causal_mask(seq_len):
    """Create causal mask for decoder self-attention"""
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask.unsqueeze(0)  # (1, seq_len, seq_len)

torch.manual_seed(42)
decoder_input = torch.randn(2, 6, 64)
encoder_output = torch.randn(2, 8, 64)
decoder = TransformerDecoder(num_layers=2, d_model=64, num_heads=8, d_ff=256)
output, (self_weights, cross_weights) = decoder(
    decoder_input, encoder_output, create_causal_mask(6))
print(f"Decoder output shape: {output.shape}")
print(f"Number of self-attention weight tensors: {len(self_weights)}")
print(f"Self-attention weights shape: {self_weights[0].shape}")
print(f"Number of cross-attention weight tensors: {len(cross_weights)}")
print(f"Cross-attention weights shape: {cross_weights[0].shape}")
